In [ ]:

import duckdb
import pandas as pd

con = duckdb.connect(r"", read_only=True)

df = con.execute("""
    SELECT *
    FROM modeling.home_credit_dataset
""").df()

print("Dataset shape:", df.shape)
df.head()



Dataset shape: (356255, 93)


,SK_ID_CURR,TARGET,dataset_split,registration_years,income_per_person,residual_income_ratio,annuity_ratio,log_income,personal_leverage,goods_income_ratio,...,cc_util_mean_12m,cc_util_slope,cc_util_std,cc_util_range,cc_balance_cv,n_months_util_gt80,cc_util_3m_minus_lifetime,cc_util_6m_minus_12m,max_consec_util_gt80,max_consec_util_gt90
0,213749,0,train,10.431211,94500.0,0.828262,0.171738,12.149508,4.762190,3.690476,...,0.516404,0.001982,0.399885,1.028595,1.144474,10.0,0.658522,0.448206,6,5
1,454959,<NA>,test,20.509240,66000.0,0.815205,0.184795,12.196027,2.655227,2.272727,...,0.000000,-0.004521,0.054089,0.209596,3.840914,0.0,-0.014082,0.000000,0,0
2,232392,<NA>,test,1.905544,94500.0,0.863310,0.136690,12.149508,2.128119,1.738095,...,0.933787,0.036181,0.397904,1.014263,0.569254,20.0,0.161012,-0.047665,18,10
3,306464,<NA>,test,25.062286,56250.0,0.896840,0.103160,11.630717,1.417960,1.200000,...,NaN,0.000000,0.000000,0.000000,NaN,0.0,NaN,NaN,0,0
4,444744,0,train,15.819302,60000.0,0.772400,0.227600,12.100718,4.195500,3.750000,...,0.818524,-0.001398,0.273028,1.072359,0.382007,41.0,0.299903,0.071217,14,9


In [2]:
# Group By and Formatted Output
target_summary = df.groupby("dataset_split").agg(
    total_rows=("SK_ID_CURR", "size"),
    labeled_customers=("TARGET", "count"),
    default_rate=("TARGET", "mean")
).reset_index()

target_summary["default_rate"] = (
    target_summary["default_rate"] * 100
).round(2)

target_summary = target_summary.rename(columns={
    "dataset_split": "Dataset",
    "total_rows": "Total rows",
    "labeled_customers": "Labeled customers",
    "default_rate": "Observed default rate (%)"
})

print(target_summary.to_string(index=False, na_rep="N/A"))

Dataset  Total rows  Labeled customers  Observed default rate (%)
   test       48744                  0                       <NA>
  train      307511             307511                       8.07


In [3]:
# Missing Value per data row
missing_summary = (
    df.isna().mean().mul(100).sort_values(ascending=False).to_frame(name="missing_pct")
)

print(missing_summary.to_string())


                            missing_pct
cc_balance_cv                 80.337960
cc_util_3m_minus_lifetime     78.175464
cc_util_mean_3m               78.175464
cc_util_mean_6m               77.215478
cc_util_6m_minus_12m          77.215478
cc_util_mean_12m              76.174089
cc_util_std                   71.437313
cc_util_slope                 71.437313
cc_util_mean                  71.243912
cc_util_last                  71.243912
cc_util_gt90_rate             71.243912
cc_util_median                71.243912
cc_util_max                   71.243912
cc_util_p95                   71.243912
cc_util_p75                   71.243912
cc_max_overlimit_ratio        71.243912
cc_util_gt80_rate             71.243912
cc_util_range                 71.243912
cc_dpd_rate_6m                70.932057
cc_diff_dpd_rate_6m_12m       70.932057
cc_dpd_rate                   70.931496
cc_max_dpd                    70.931496
cc_dpd_rate_12m               70.931496
max_consec_util_gt90          70.931496


In [4]:

import numpy as np

condition = df["dataset_split"] == "train"
train_df = df[condition].copy()

features = train_df.drop(columns=["SK_ID_CURR", "TARGET", "dataset_split"])

quality_report = pd.DataFrame({
    "dtype": features.dtypes,
    "missing_pct": features.isna().mean() * 100,
    "unique_values": features.nunique(dropna=True)
})

numeric = features.select_dtypes(include="number")

infinity_flags = numeric.isin([np.inf, -np.inf])
infinity_counts = infinity_flags.sum()
quality_report.loc[numeric.columns, "inf_count"] = infinity_counts

numeric_clean = numeric.replace([np.inf, -np.inf], np.nan)

quality_report.loc[numeric.columns, "min"] = numeric_clean.min()
quality_report.loc[numeric.columns, "p01"] = numeric_clean.quantile(0.01)
quality_report.loc[numeric.columns, "median"] = numeric_clean.median()
quality_report.loc[numeric.columns, "p99"] = numeric_clean.quantile(0.99)
quality_report.loc[numeric.columns, "max"] = numeric_clean.max()

quality_report 



,dtype,missing_pct,unique_values,inf_count,min,p01,median,p99,max
registration_years,float64,0.000000,15688,0,-0.0,0.136893,12.33128,37.998631,67.548255
income_per_person,float64,0.000650,3064,0,2812.5,18000.0,75000.0,337500.0,39000000.0
residual_income_ratio,float64,0.003902,88811,0,-0.875965,0.516542,0.837167,0.960526,0.999776
annuity_ratio,float64,0.003902,88811,0,0.000224,0.039474,0.162833,0.483458,1.875965
log_income,float64,0.000000,2548,0,10.152338,10.71444,11.899215,13.065795,18.577685
...,...,...,...,...,...,...,...,...,...
n_months_util_gt80,float64,71.739222,97,0,0.0,0.0,2.0,70.0,96.0
cc_util_3m_minus_lifetime,float64,78.734419,43949,0,-1.087329,-0.687275,0.0,0.624756,6.843516
cc_util_6m_minus_12m,float64,77.900303,33207,0,-0.899148,-0.369541,0.0,0.346961,1.779116
max_consec_util_gt80,Int64,71.739222,97,0,0.0,0.0,2.0,52.0,96.0


In [5]:

inf_count_columns = quality_report[quality_report["inf_count"] > 0 ]
print (f"Columns with inf value", inf_count_columns.index.tolist())

constant_features = quality_report[quality_report["unique_values"] <= 1] 
print (f"Constant features: ", constant_features.index.tolist())



Columns with inf value []
Constant features:  []


In [6]:
cols = ["employment_age_ratio", "employment_years"]

quality_report.loc[cols,
    ["missing_pct", "min", "p01", "median", "p99", "max"]
]

,missing_pct,min,p01,median,p99,max
employment_age_ratio,18.007161,0.0,0.007548,0.118733,0.5906,0.728811
employment_years,18.007161,0.0,0.303901,4.511978,31.042519,49.040383


In [ ]:
# Ratio check
bounded_features = [
    "employment_age_ratio",
    "housing_info_missing_rate",
    "inst_late_rate",
    "dpd_month_rate",
    "pos_dpd_rate",
    "pos_dpd_rate_6m",
    "pos_dpd_rate_12m",
    "cc_dpd_rate",
    "cc_dpd_rate_6m",
    "cc_dpd_rate_12m",
    "pay_underpay_rate",
    "pay_underpay_rate_6m", 
    "pay_shortfall_ratio",
    "cc_util_gt80_rate",
    "cc_util_gt90_rate",
    "cc_positive_balance_rate"
]

cols = []
for c in bounded_features:
    if c in train_df.columns: 
        cols.append(c)

domain_report = pd.DataFrame({
    "min": train_df[cols].min(),
    "max": train_df[cols].max(),
    "invalid_count": (  (train_df[cols] < 0) |
                        (train_df[cols] > 1) ).sum() 
                            })
domain_report.sort_values("invalid_count",ascending=False)



,min,max,invalid_count
employment_age_ratio,0.0,0.728811,0
housing_info_missing_rate,0.0,1.000000,0
inst_late_rate,0.0,1.000000,0
dpd_month_rate,0.0,1.000000,0
pos_dpd_rate,0.0,1.000000,0
pos_dpd_rate_6m,0.0,1.000000,0
pos_dpd_rate_12m,0.0,1.000000,0
cc_dpd_rate,0.0,0.989474,0
cc_dpd_rate_6m,0.0,1.000000,0
cc_dpd_rate_12m,0.0,1.000000,0


In [8]:
# Non negtive check 
nonnegative_features = [
    "age_years",
    "child_burden",
    "employment_years",
    "bur_account_count",
    "bur_active_count",
    "bur_active_debt",
    "bur_total_debt",
    "bur_total_credit",
    "bur_total_overdue",
    "bur_max_debt",
    "bur_new_credit_12m",
    "bur_days_since_last_credit",
    "bur_oldest_credit_days",
    "asset_count",
    "housing_info_count",
    "max_early_days",
    "max_late_days",
    "pos_max_dpd",
    "cc_max_dpd",
    "inst_obs_count",
    "pay_max_shortfall",
    "n_months_util_gt80",
    "max_consec_util_gt80",
    "max_consec_util_gt90"
]

cols_2 = []
for c in nonnegative_features:
    if c in train_df.columns:
        cols_2.append(c)

nonnegative_report = pd.DataFrame({
    "min":train_df[cols_2].min(),
    "max":train_df[cols_2].max(),
    "negative_count": (train_df[cols_2] < 0).sum(),
    "missing_count": ((train_df[cols_2] < 0).isna()).sum(),
    "missing_pct": ((train_df[cols_2] < 0).isna()).mean()
                                    })

nonnegative_report.sort_values("missing_pct", ascending=False)

,min,max,negative_count,missing_count,missing_pct
max_consec_util_gt90,0.0,95.0,0,220606,0.717392
max_consec_util_gt80,0.0,96.0,0,220606,0.717392
cc_max_dpd,0.0,2831.0,0,220606,0.717392
bur_days_since_last_credit,0.0,2922.0,0,44020,0.143149
bur_oldest_credit_days,0.0,2922.0,0,44020,0.143149
pos_max_dpd,0.0,4231.0,0,18067,0.058752
inst_obs_count,1.0,332.0,0,15868,0.051601
bur_active_debt,-6978830.49,334498331.205,1091,0,0.000000
max_early_days,0.0,3189.0,0,0,0.000000
employment_years,0.0,49.040383,0,0,0.000000


In [9]:

from sklearn.model_selection import train_test_split

train_df = df[df["dataset_split"] == "train"].copy()

assert train_df["TARGET"].notna().all()
assert train_df["TARGET"].isin([0, 1]).all()
assert train_df["SK_ID_CURR"].is_unique

X = train_df.drop(
    columns=["SK_ID_CURR", "TARGET", "dataset_split"]
)

y = train_df["TARGET"].astype(int)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Validation:", X_valid.shape)

print("Train default rate:", y_train.mean())
print("Validation default rate:", y_valid.mean())


Train: (246008, 90)
Validation: (61503, 90)
Train default rate: 0.08072908198107379
Validation default rate: 0.08072776937710356


In [10]:

import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_cols = X_train.select_dtypes(include="number").columns.tolist()

categorical_cols = X_train.select_dtypes(exclude="number").columns.tolist()

print("Numerical:", len(numeric_cols))
print("Categorical:", len(categorical_cols))
print("Total:", len(numeric_cols) + len(categorical_cols))

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(  strategy="median",
                                add_indicator=True
    )),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Unknown"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])


Numerical: 84
Categorical: 6
Total: 90


In [11]:

X_train_ready = preprocessor.fit_transform(X_train)

X_valid_ready = preprocessor.transform(X_valid)

print("Before:", X_train.shape)
print("After:", X_train_ready.shape)

print("Validation:", X_valid_ready.shape)


Before: (246008, 90)
After: (246008, 250)
Validation: (61503, 250)


In [12]:

feature_names = preprocessor.get_feature_names_out()

print("Number of transformed features:", len(feature_names))
print("First 20 features:")

print(feature_names[:20])

print("Output type:", type(X_train_ready))


Number of transformed features: 250
First 20 features:
['num__registration_years' 'num__income_per_person'
 'num__residual_income_ratio' 'num__annuity_ratio' 'num__log_income'
 'num__personal_leverage' 'num__goods_income_ratio' 'num__age_years'
 'num__child_burden' 'num__employment_years' 'num__employment_age_ratio'
 'num__employment_unknown_flag' 'num__bur_account_count'
 'num__bur_active_count' 'num__bur_active_debt' 'num__bur_total_debt'
 'num__bur_total_credit' 'num__bur_total_overdue' 'num__bur_total_limit'
 'num__bur_max_debt']
Output type: <class 'numpy.ndarray'>


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

baseline_model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

baseline_model.fit(X_train, y_train)

y_pred_proba = baseline_model.predict_proba(X_valid)[:, 1]

roc_auc = roc_auc_score(y_valid, y_pred_proba)
pr_auc = average_precision_score(y_valid, y_pred_proba)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Average Precision: {pr_auc:.4f}")


ROC-AUC: 0.7084
Average Precision: 0.1917


In [14]:

eval_df = pd.DataFrame({
    "actual": y_valid.to_numpy(),
    "risk_score": y_pred_proba
})

eval_df["risk_decile"] = pd.qcut(
    eval_df["risk_score"].rank(method="first"),
    q=10,
    labels=False
) + 1

decile_report = eval_df.groupby("risk_decile").agg(
    customers=("actual", "size"),
    defaults=("actual", "sum"),
    mean_score=("risk_score", "mean"),
    default_rate=("actual", "mean")
)

overall_default_rate = eval_df["actual"].mean()
total_defaults = eval_df["actual"].sum()

decile_report["default_rate_pct"] = (
    decile_report["default_rate"] * 100
)

decile_report["default_capture_pct"] = (
    decile_report["defaults"] / total_defaults * 100
)

decile_report["lift"] = (
    decile_report["default_rate"] / overall_default_rate
)

display(
    decile_report.sort_index(ascending=False).round(3)
)


,customers,defaults,mean_score,default_rate,default_rate_pct,default_capture_pct,lift
risk_decile,,,,,,,
10,6151,1402,0.231,0.228,22.793,28.238,2.823
9,6150,818,0.135,0.133,13.301,16.475,1.648
8,6150,647,0.103,0.105,10.520,13.031,1.303
7,6150,516,0.083,0.084,8.390,10.393,1.039
6,6150,409,0.069,0.067,6.650,8.238,0.824
5,6151,334,0.057,0.054,5.430,6.727,0.673
4,6150,294,0.047,0.048,4.780,5.921,0.592
3,6150,215,0.039,0.035,3.496,4.330,0.433
2,6150,205,0.030,0.033,3.333,4.129,0.413


In [15]:

lr_train_proba = baseline_model.predict_proba(X_train)[:, 1]

print("LR Train AUC:",
      roc_auc_score(y_train, lr_train_proba))

print("LR Valid AUC:",
      roc_auc_score(y_valid, y_pred_proba))


LR Train AUC: 0.7159677635644
LR Valid AUC: 0.708416683240642


In [16]:

import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

# Check transformed inputs
print("Train finite:", np.isfinite(X_train_ready).all())
print("Valid finite:", np.isfinite(X_valid_ready).all())

# Other ML model
hgb_model = HistGradientBoostingClassifier(
    max_iter=120,
    learning_rate=0.05,
    max_leaf_nodes=15,
    min_samples_leaf=100,
    l2_regularization=5.0,
    early_stopping=True,
    random_state=42
)

hgb_model.fit(X_train_ready, y_train)

hgb_train_proba = hgb_model.predict_proba(
    X_train_ready
)[:, 1]

hgb_valid_proba = hgb_model.predict_proba(
    X_valid_ready
)[:, 1]

# Evaluation
print(
    "Train ROC-AUC:",
    roc_auc_score(y_train, hgb_train_proba)
)

print(
    "Valid ROC-AUC:",
    roc_auc_score(y_valid, hgb_valid_proba)
)

print(
    "Valid AP:",
    average_precision_score(y_valid, hgb_valid_proba)
)


Train finite: True
Valid finite: True
Train ROC-AUC: 0.7303882060604967
Valid ROC-AUC: 0.7130808189784539
Valid AP: 0.1995057848872877


In [17]:

import pandas as pd

hgb_eval = pd.DataFrame({
    "actual": y_valid.to_numpy(),
    "risk_score": hgb_valid_proba
})

hgb_eval["risk_decile"] = pd.qcut(
    hgb_eval["risk_score"].rank(method="first"),
    q=10,
    labels=False
) + 1

hgb_decile = hgb_eval.groupby("risk_decile").agg(
    customers=("actual", "size"),
    defaults=("actual", "sum"),
    mean_score=("risk_score", "mean"),
    default_rate=("actual", "mean")
)

overall_rate = hgb_eval["actual"].mean()
total_defaults = hgb_eval["actual"].sum()

hgb_decile["default_rate_pct"] = (
    hgb_decile["default_rate"] * 100
)

hgb_decile["capture_pct"] = (
    hgb_decile["defaults"] / total_defaults * 100
)

hgb_decile["lift"] = (
    hgb_decile["default_rate"] / overall_rate
)

display(
    hgb_decile.sort_index(ascending=False).round(3)
)


,customers,defaults,mean_score,default_rate,default_rate_pct,capture_pct,lift
risk_decile,,,,,,,
10,6151,1450,0.213,0.236,23.573,29.204,2.920
9,6150,812,0.130,0.132,13.203,16.354,1.636
8,6150,655,0.100,0.107,10.650,13.192,1.319
7,6150,515,0.082,0.084,8.374,10.373,1.037
6,6150,380,0.069,0.062,6.179,7.654,0.765
5,6151,336,0.059,0.055,5.463,6.767,0.677
4,6150,265,0.051,0.043,4.309,5.337,0.534
3,6150,244,0.044,0.040,3.967,4.914,0.491
2,6150,177,0.036,0.029,2.878,3.565,0.357


In [18]:


y_true = y_valid.to_numpy()

predictions = {
    "Logistic Regression": baseline_model.predict_proba(X_valid)[:, 1],
    "HistGradientBoosting": hgb_valid_proba
}

rng = np.random.default_rng(42)
n_boot = 300

boot_results = {
    model: {"ROC_AUC": [], "AP": []}
    for model in predictions
}

for i in range(n_boot):
    idx = rng.integers(0, len(y_true), len(y_true))
    y_sample = y_true[idx]

    if len(np.unique(y_sample)) < 2:
        continue

    for model, proba in predictions.items():
        p_sample = proba[idx]

        boot_results[model]["ROC_AUC"].append(
            roc_auc_score(y_sample, p_sample)
        )
        boot_results[model]["AP"].append(
            average_precision_score(y_sample, p_sample)
        )

rows = []

for model, proba in predictions.items():
    actual = {
        "ROC_AUC": roc_auc_score(y_true, proba),
        "AP": average_precision_score(y_true, proba)
    }

    for metric, estimate in actual.items():
        scores = np.array(boot_results[model][metric])

        rows.append({
            "Model": model,
            "Metric": metric,
            "Estimate": estimate,
            "Bootstrap_SE": scores.std(ddof=1),
            "CI95_Lower": np.percentile(scores, 2.5),
            "CI95_Upper": np.percentile(scores, 97.5),
            "P05": np.percentile(scores, 5),
            "P95": np.percentile(scores, 95)
        })

performance_table = pd.DataFrame(rows)

display(performance_table.round(4))


,Model,Metric,Estimate,Bootstrap_SE,CI95_Lower,CI95_Upper,P05,P95
0,Logistic Regression,ROC_AUC,0.7084,0.0039,0.7002,0.7154,0.7016,0.7144
1,Logistic Regression,AP,0.1917,0.0052,0.1822,0.2025,0.1835,0.2009
2,HistGradientBoosting,ROC_AUC,0.7131,0.0039,0.7048,0.7201,0.7060,0.7187
3,HistGradientBoosting,AP,0.1995,0.0053,0.1896,0.2104,0.1908,0.2081


In [19]:
lr_classifier = baseline_model.named_steps["classifier"]
lr_preprocessor = baseline_model.named_steps["preprocessing"]

feature_names = lr_preprocessor.get_feature_names_out()

lr_coef_table = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": lr_classifier.coef_[0]
})

lr_coef_table["Odds_Ratio"] = np.exp(
    lr_coef_table["Coefficient"]
)

lr_coef_table["Abs_Coefficient"] = (
    lr_coef_table["Coefficient"].abs()
)

lr_coef_table = lr_coef_table.sort_values(
    "Abs_Coefficient",
    ascending=False
)

display(lr_coef_table.head(20).round(4))


,Feature,Coefficient,Odds_Ratio,Abs_Coefficient
6,num__goods_income_ratio,-0.9871,0.3727,0.9871
5,num__personal_leverage,0.8796,2.4100,0.8796
236,cat__ORGANIZATION_TYPE_Transport: type 3,0.5915,1.8068,0.5915
242,cat__FLAG_OWN_CAR_N,-0.5853,0.5569,0.5853
241,cat__FLAG_OWN_REALTY_Y,-0.5718,0.5645,0.5718
240,cat__FLAG_OWN_REALTY_N,-0.5018,0.6055,0.5018
163,cat__OCCUPATION_TYPE_Accountants,-0.4974,0.6081,0.4974
243,cat__FLAG_OWN_CAR_Y,-0.4883,0.6137,0.4883
172,cat__OCCUPATION_TYPE_Low-skill Laborers,0.4560,1.5777,0.4560
213,cat__ORGANIZATION_TYPE_Military,-0.4178,0.6585,0.4178


In [21]:
import statsmodels.api as sm

selected_features = [
    "age_years",
    "log_income",
    "annuity_ratio",
    "employment_years",
    "employment_unknown_flag",
    "bur_account_count",
    "pos_dpd_rate",
    "cc_util_mean"
]

X_sm = X_train[selected_features].copy()

X_sm["cc_util_missing"] = (
    X_sm["cc_util_mean"].isna().astype(int)
)

continuous_cols = [
    c for c in selected_features
    if c != "employment_unknown_flag"
]

medians = X_sm[continuous_cols].median()
X_sm[continuous_cols] = (
    X_sm[continuous_cols].fillna(medians)
)

scaler_sm = StandardScaler()
X_sm[continuous_cols] = scaler_sm.fit_transform(
    X_sm[continuous_cols]
)

X_sm = sm.add_constant(
    X_sm.astype(float),
    has_constant="add"
)

inference_model = sm.GLM(
    y_train.astype(float),
    X_sm,
    family=sm.families.Binomial()
)

inference_result = inference_model.fit(
    cov_type="HC3",
    maxiter=100
)

print(inference_result.summary())



                 Generalized Linear Model Regression Results                  
Dep. Variable:                 TARGET   No. Observations:               246008
Model:                            GLM   Df Residuals:                   245998
Model Family:                Binomial   Df Model:                            9
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -67047.
Date:                Sun, 25 Oct 2026   Deviance:                   1.3409e+05
Time:                        14:43:49   Pearson chi2:                 2.47e+05
No. Iterations:                     6   Pseudo R-squ. (CS):            0.01588
Covariance Type:                  HC3                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

In [22]:

from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

hgb_explainer = Pipeline([
    ("preprocessing",
     baseline_model.named_steps["preprocessing"]),
    ("classifier", hgb_model)
])

np.testing.assert_allclose(
    hgb_explainer.predict_proba(X_valid)[:, 1],
    hgb_valid_proba,
    rtol=1e-6,
    atol=1e-8
)

perm_result = permutation_importance(
    hgb_explainer,
    X_valid,
    y_valid,
    scoring="roc_auc",
    n_repeats=3,
    max_samples=2000,
    random_state=42,
    n_jobs=1
)

hgb_importance_table = pd.DataFrame({
    "Feature": X_valid.columns,
    "Importance_Mean": perm_result.importances_mean,
    "Importance_SD": perm_result.importances_std
})

hgb_importance_table = hgb_importance_table.sort_values(
    "Importance_Mean",
    ascending=False
)

display(hgb_importance_table.head(20).round(4))


,Feature,Importance_Mean,Importance_SD
27,debt_credit,-0.0199,0.0034
58,inst_obs_count,-0.0257,0.0027
10,OCCUPATION_TYPE,-0.0312,0.0023
24,bur_days_since_last_credit,-0.0317,0.0014
7,age_years,-0.0323,0.0036
5,personal_leverage,-0.0328,0.0016
36,OWN_CAR_AGE,-0.0344,0.0021
12,employment_years,-0.0347,0.0044
23,bur_oldest_credit_days,-0.0351,0.0042
9,NAME_INCOME_TYPE,-0.0366,0.0016
